# QDMR on Pilot 2 benchmark samples

Loads a few comparative questions from `data/pilot2_benchmark_english.v1.json`.

Order of sections:

1. **Gold** — sub-questions in the JSON
2. **Break** — fine-tuned draft
3. **LLM** — DSPy rewrite (Break skeleton + wording polish)
4. **Advanced LLM (DSPy)** — multi-stage CoT + repair
5. **LLM pipeline** — full question understanding + decomposition without Break

Later steps cite earlier ones as `#k`. The last step is the synthesize node.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import BREAK_QDMR_MODEL
from src.decomposition.benchmark_fewshots import iter_decompositions
from IPython.display import HTML, display

from src.decomposition.graph_view import render_graph, render_pair
from src.decomposition.syntax_parser import parse_decomposition
from src.llm import configure_refinement_lm

In [2]:
SAMPLE_IDS = ["0", "9", "27"]  # Dresden, marriage, love
samples = [item for item in iter_decompositions() if item["question_id"] in SAMPLE_IDS]
samples.sort(key=lambda item: SAMPLE_IDS.index(item["question_id"]))
assert len(samples) == len(SAMPLE_IDS)

for sample in samples:
    print(f"===== Q{sample['question_id']} {sample['concept']} =====")
    print("main question:", sample["question"])
    print("years:", sample["years"])
    print()

===== Q0 Dresden =====
main question: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
years: [1830, 1875, 2025]

===== Q9 Marriage =====
main question: How does the definition of a valid marriage shift over time in the Western hemisphere?
years: [1830, 1875, 2025]

===== Q27 Love =====
main question: What are conceptual differences between love in the early 19th century and today?
years: [1830, 2025]



## Sub-questions from the JSON

These are the benchmark's own decompositions, rewritten as a Break string. The last step asks how the accounts differ and cites the earlier sub-questions.

In [3]:
for sample in samples:
    print(f"===== Q{sample['question_id']} gold =====")
    print("main question:", sample["question"])
    print("sub-questions:")
    for index, question in enumerate(sample["subquestions"], start=1):
        print(f"  #{index} {question}")
    print(sample["decomposition"])
    print()

===== Q0 gold =====
main question: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
sub-questions:
  #1 How is Dresden defined?
  #2 What terms, nicknames, or descriptive phrases are used for Dresden?
  #3 What reasons, attributes, or comparisons are provided in the entries to explain Dresden's cultural reputation?
return How is Dresden defined? ;return What terms, nicknames, or descriptive phrases are used for Dresden? ;return What reasons, attributes, or comparisons are provided in the entries to explain Dresden's cultural reputation? ;return How do these accounts differ, given #1, #2, #3?

===== Q9 gold =====
main question: How does the definition of a valid marriage shift over time in the Western hemisphere?
sub-questions:
  #1 How is marriage defined?
  #2 What is the legal age to marry in different sources?
  #3 Which requirements need to be in place for a marriage to happen?
  #4 How can a marriage be dissolved c

## Break model

The fine-tuned checkpoint `mrm8488/t5-base-finetuned-break_data` drafts the plan. Short Break phrases such as `return marriage` are expanded into readable sub-questions before the graph is drawn. The model raw string is still printed for inspection.

The graph reads top to bottom. An arrow means the lower step cites the upper one as `#k`. The last step is the synthesize node.

In [4]:
import os

from src.decomposition.backends.finetuned_backend import FineTunedQDMRBackend

model_path = os.getenv("QDMR_MODEL_NAME_OR_PATH", BREAK_QDMR_MODEL).strip() or BREAK_QDMR_MODEL
print("checkpoint:", model_path)
break_model = FineTunedQDMRBackend(model_name_or_path=model_path, max_new_tokens=128)
print("loaded:", not break_model.uses_mock)
if break_model._load_error:
    print("error:", break_model._load_error)

break_drafts = []
for sample in samples:
    steps = break_model.decompose(sample["question"], sample["slices"], sample["concept"])
    break_drafts.append(
        {
            "raw": break_model.last_raw,
            "model_raw": break_model.last_model_raw,
            "steps": steps,
        }
    )
    print(f"\n===== Q{sample['question_id']} {sample['concept']} =====")
    print("main question:", sample["question"])
    print("model raw:", break_model.last_model_raw)
    print("readable draft:", break_model.last_raw)
    display(HTML(render_graph(steps, "Break draft")))

checkpoint: mrm8488/t5-base-finetuned-break_data
loaded: False
error: No module named 'torch'

===== Q0 Dresden =====
main question: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
model raw: return How is Dresden described in 1830? ;return How is Dresden described in 1875? ;return How is Dresden described in 2025? ;return How does Dresden differ across #1, #2, #3?
readable draft: return How is Dresden described in 1830? ;return How is Dresden described in 1875? ;return How is Dresden described in 2025? ;return How does Dresden differ across #1, #2, #3?



===== Q9 Marriage =====
main question: How does the definition of a valid marriage shift over time in the Western hemisphere?
model raw: return How is Marriage described in 1830? ;return How is Marriage described in 1875? ;return How is Marriage described in 2025? ;return How does Marriage differ across #1, #2, #3?
readable draft: return How is Marriage described in 1830? ;return How is Marriage described in 1875? ;return How is Marriage described in 2025? ;return How does Marriage differ across #1, #2, #3?



===== Q27 Love =====
main question: What are conceptual differences between love in the early 19th century and today?
model raw: return How is Love described in 1830? ;return How is Love described in 2025? ;return How does Love differ across #1, #2?
readable draft: return How is Love described in 1830? ;return How is Love described in 2025? ;return How does Love differ across #1, #2?


## DSPy rewrite (normal)

Break keeps the `#k` plan. A single DSPy `Predict` only rewrites wording into full sub-questions.

Uses the two nearest few-shots (never the asked question). Scayle LLM via `.env`.

In [5]:
from src.decomposition.backends.dspy_backend import DSPyQDMRRefiner
from src.decomposition.refine_utils import format_edges, score_against_gold
from src.llm import env_files

print("LLM config is environment variables, not YAML.")
print("code:", "src/llm.py")
for path in env_files():
    print("env:", path)

language_model = configure_refinement_lm(prefer="scayle", force=True)
print("language model:", language_model or "none; the Break draft is kept")

refiner = DSPyQDMRRefiner(exclude_question_ids=set(SAMPLE_IDS), max_demos=2)
rewrite_results = []
for sample, draft in zip(samples, break_drafts):
    refined = refiner.refine(
        sample["question"],
        sample["slices"],
        sample["concept"],
        draft["raw"],
        draft_steps=draft["steps"],
    )
    gold_steps = parse_decomposition(sample["decomposition"])
    print(f"\n===== Q{sample['question_id']} {sample['concept']} =====")
    print("main question:", sample["question"])
    print("Break edges:", format_edges(draft["steps"]))
    print("nearest few-shots:")
    for example in refiner.demo_questions:
        print("  -", example)
    if sample["question"] in refiner.demo_questions:
        raise RuntimeError("the asked question is inside the few-shots")
    if refined is None:
        print(f"kept the Break draft: {refiner.last_error}")
        final_steps = draft["steps"]
        final_title = "DSPy rewrite (Break draft kept)"
    else:
        final_steps = refined
        final_title = "DSPy rewrite"
    print("Final edges:", format_edges(final_steps))
    metric = score_against_gold(gold_steps, final_steps)
    print(
        "metric:",
        f"score={metric['score']}",
        f"overlap={metric['text_overlap']}",
        f"steps={metric['pred_steps']}/{metric['gold_steps']}",
        f"synthesize={metric['synthesize']}",
    )
    rewrite_results.append({"steps": final_steps, "metric": metric, "title": final_title})
    display(HTML(render_pair(gold_steps, final_steps, "Ground truth", final_title)))
    display(HTML(render_graph(draft["steps"], "Break connections kept as skeleton")))


LLM config is environment variables, not YAML.
code: src/llm.py
env: /Users/noor/Projects/DataGEMS/dg-query-disambiguation/.env
language model: scayle:qwen3



===== Q0 Dresden =====
main question: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
Break edges: #1  #2  #3  #4←#1,#2,#3
nearest few-shots:
  - How do historical accounts differ in reporting Galileo’s origin and family?
  - How do accounts differ about whether philosophy historically included the natural sciences and how they describe the process by which those sciences became separated from "philosophy"?
Final edges: #1  #2  #3  #4←#1,#2,#3
metric: score=0.787 overlap=0.15 steps=4/4 synthesize=True



===== Q9 Marriage =====
main question: How does the definition of a valid marriage shift over time in the Western hemisphere?
Break edges: #1  #2  #3  #4←#1,#2,#3
nearest few-shots:
  - How do accounts differ about whether philosophy historically included the natural sciences and how they describe the process by which those sciences became separated from "philosophy"?
  - How do historical accounts differ in reporting Galileo’s origin and family?
Final edges: #1  #2  #3  #4←#1,#2,#3
metric: score=0.649 overlap=0.194 steps=4/6 synthesize=True



===== Q27 Love =====
main question: What are conceptual differences between love in the early 19th century and today?
Break edges: #1  #2  #3←#1,#2
nearest few-shots:
  - How do historical accounts vary in tracing the origin of "socialism"?
  - How was Odin portrayed in earlier records compared to today?
Final edges: #1  #2  #3←#1,#2
metric: score=0.637 overlap=0.147 steps=3/4 synthesize=True


## Advanced DSPy (after normal rewrite)

Multi-stage `ChainOfThought`: readable per-slice sub-questions → Break DAG → validate/repair.
Uses nearest few-shots and the Break draft as an edge hint.

Shown **after** the normal DSPy rewrite so you can compare: gold | rewrite, then rewrite | advanced.

In [6]:
from pathlib import Path

from src.decomposition.advanced_dspy_module import AdvancedQDMRDecomposer, load_compiled_state
from src.decomposition.validation import validate_qdmr_graph

if configure_refinement_lm(prefer="scayle", force=False) is None:
    print("no language model configured; advanced falls back to stubs")

compiled = (
    os.getenv("QDMR_COMPILED_PATH", "").strip()
    or str(ROOT / "artifacts" / "compiled_qdmr.json")
)
advanced = AdvancedQDMRDecomposer(
    max_assertion_retries=3,
    exclude_question_ids=set(SAMPLE_IDS),
    max_demos=2,
)
if Path(compiled).is_file() and Path(compiled).stat().st_size > 2:
    load_compiled_state(advanced, compiled)
    print("compiled weights:", compiled)
else:
    print("compiled weights: none (base prompts + few-shots)")

for sample, draft, rewrite in zip(samples, break_drafts, rewrite_results):
    gold_steps = parse_decomposition(sample["decomposition"])
    advanced_steps = advanced.decompose(
        sample["question"],
        sample["slices"],
        sample["concept"],
        draft=draft["raw"],
        draft_steps=draft["steps"],
    )
    goals = advanced.program.last_goals
    raw = advanced.last_raw
    repairs = advanced.assertion_repairs
    valid, message = validate_qdmr_graph(
        raw or " ;".join(f"return {step.instruction}" for step in advanced_steps),
        expected_min_steps=len(sample["slices"]) + 1,
    )
    rewrite_metric = rewrite["metric"]
    advanced_metric = score_against_gold(gold_steps, advanced_steps)

    print(f"\n===== Q{sample['question_id']} {sample['concept']} =====")
    print("main question:", sample["question"])
    print("few-shots:")
    for example in advanced.demo_questions:
        print("  -", example)
    print("planned sub-questions:")
    for index, goal in enumerate(goals, start=1):
        print(f"  {index}. {goal}")
    if advanced.last_error:
        print("note:", advanced.last_error)
    print("assertion repairs:", repairs, "| valid:", valid if valid else message)
    print("Rewrite edges: ", format_edges(rewrite["steps"]))
    print("Advanced edges:", format_edges(advanced_steps))
    print(
        "rewrite metric: ",
        f"score={rewrite_metric['score']}",
        f"overlap={rewrite_metric['text_overlap']}",
        f"steps={rewrite_metric['pred_steps']}/{rewrite_metric['gold_steps']}",
    )
    print(
        "advanced metric:",
        f"score={advanced_metric['score']}",
        f"overlap={advanced_metric['text_overlap']}",
        f"steps={advanced_metric['pred_steps']}/{advanced_metric['gold_steps']}",
    )
    if raw:
        print("advanced raw:", raw)

    display(HTML(render_pair(rewrite["steps"], advanced_steps, "DSPy rewrite (normal)", "Advanced DSPy")))
    display(HTML(render_pair(gold_steps, advanced_steps, "Ground truth", "Advanced DSPy")))


compiled weights: none (base prompts + few-shots)



===== Q0 Dresden =====
main question: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
few-shots:
  - How do historical accounts differ in reporting Galileo’s origin and family?
  - How do accounts differ about whether philosophy historically included the natural sciences and how they describe the process by which those sciences became separated from "philosophy"?
planned sub-questions:
  1. What cultural nicknames were given to Dresden in 1830, and what reasons were provided for them?
  2. What cultural nicknames were given to Dresden in 1875, and what reasons were provided for them?
  3. What cultural nicknames were given to Dresden in 2025, and what reasons were provided for them?
  4. How do the cultural nicknames and the reasons for them differ for Dresden across 1830, 1875, and 2025?
assertion repairs: 0 | valid: True
Rewrite edges:  #1  #2  #3  #4←#1,#2,#3
Advanced edges: #1  #2  #3  #4←#1,#2,#3
rewrite metric: 


===== Q9 Marriage =====
main question: How does the definition of a valid marriage shift over time in the Western hemisphere?
few-shots:
  - How do accounts differ about whether philosophy historically included the natural sciences and how they describe the process by which those sciences became separated from "philosophy"?
  - How do historical accounts differ in reporting Galileo’s origin and family?
planned sub-questions:
  1. How is the definition of a valid marriage described in the Western hemisphere in 1830?
  2. How is the definition of a valid marriage described in the Western hemisphere in 1875?
  3. How is the definition of a valid marriage described in the Western hemisphere in 2025?
  4. How does the definition of a valid marriage differ across 1830, 1875, and 2025 in the Western hemisphere?
assertion repairs: 0 | valid: True
Rewrite edges:  #1  #2  #3  #4←#1,#2,#3
Advanced edges: #1  #2  #3  #4←#1,#2,#3
rewrite metric:  score=0.649 overlap=0.194 steps=4/6
advanced metric


===== Q27 Love =====
main question: What are conceptual differences between love in the early 19th century and today?
few-shots:
  - How do historical accounts vary in tracing the origin of "socialism"?
  - How was Odin portrayed in earlier records compared to today?
planned sub-questions:
  1. How is the concept of love described in 1830?
  2. How is the concept of love described in 2025?
  3. How does the concept of love differ between 1830 and 2025?
assertion repairs: 0 | valid: True
Rewrite edges:  #1  #2  #3←#1,#2
Advanced edges: #1  #2  #3←#1,#2
rewrite metric:  score=0.637 overlap=0.147 steps=3/4
advanced metric: score=0.645 overlap=0.181 steps=3/4
advanced raw: return How is the concept of love described in 1830? ;return How is the concept of love described in 2025? ;return How do these accounts differ, given #1, #2?


## LLM pipeline — question understanding + decomposition

Runs the full pipeline **without Break** — every step uses the configured LLM:

1. **Route** — extract the target concept
2. **Disambiguate** — parse slices (regex + LLM fallback for concept)
3. **Extend knowledge** — facets, lemmas, comparative lenses via LLM
4. **Decompose** — LLM-only sub-questions (DSPy backend, no Break draft)

Configure `LLM_CHOICE` below to switch provider (`scayle`, `ollama`, `openai`).

In [7]:
import json
import os
import time

from src.decomposition.factory import QDMRDecomposerFactory
from src.knowledge_extension.expander import expand_knowledge, knowledge_extension_payload
from src.llm import configure_refinement_lm, llm_endpoint
from src.routing.router import route_query
from src.schemas.slice import parse_query_intent

# ── provider selection (change to "ollama" or "openai" as needed) ──
LLM_CHOICE = None  # None ⇒ reads LLM_PROVIDER from .env (default: scayle)

label = configure_refinement_lm(prefer=LLM_CHOICE, force=True)
print(f"LLM : {label}")
print(f"URL : {llm_endpoint()}")

os.environ["QDMR_BACKEND"] = "dspy"

# ── run the pipeline for each benchmark sample ──
for sample in samples:
    qid = sample["question_id"]
    question = sample["question"]
    print(f"\n{'─'*60}")
    print(f"  Q{qid}: {question}")
    print(f"{'─'*60}")

    # Step 1 — Route
    t0 = time.time()
    decision = route_query(question)
    llm_concept = decision.concept or ""
    print(f"  1. Route ({time.time()-t0:.1f}s): concept = {llm_concept or '(regex only)'}")

    # Step 2 — Disambiguate
    intent, errors = parse_query_intent(question)
    if intent is None and llm_concept:
        patched = f"How were descriptions of {llm_concept} different {question.split('in ', 1)[-1] if ' in ' in question else question}"
        intent, errors = parse_query_intent(patched)
        if intent:
            intent.original_query = question
            intent.target_concept = llm_concept
            print(f"  2. Disambiguate: concept from LLM → {llm_concept}")
    if intent is None:
        print(f"  ⚠ disambiguation failed: {errors}")
        continue
    concept = intent.target_concept
    print(f"  2. Disambiguate: {concept}, {intent.dimension}, {len(intent.slices)} slices")
    for s in intent.slices:
        print(f"       {s.slice_id}: {s.label} ({s.period_start}–{s.period_end}, {s.language})")

    # Step 3 — Extend knowledge
    t0 = time.time()
    expansion = expand_knowledge(question, concept, intent.slices)
    payload = knowledge_extension_payload(expansion)
    print(f"  3. Extend ({time.time()-t0:.1f}s):")
    print(f"       facets: {payload.get('thematic_facets', [])}")
    print(f"       lenses: {payload.get('feature_lenses', [])}")

    # Step 4 — Decompose (LLM only)
    t0 = time.time()
    decomposer = QDMRDecomposerFactory.create(backend="dspy")
    steps = decomposer.decompose(question, intent.slices, concept)
    fallback = getattr(decomposer, "used_fallback", False)
    tag = " (fallback)" if fallback else ""
    print(f"  4. Decompose ({time.time()-t0:.1f}s){tag}:")
    for st in steps:
        refs = f"  → {st.references}" if st.references else ""
        print(f"       {st.instruction}{refs}")

print(f"\n{'═'*60}")
print("  Pipeline finished for all benchmark samples.")

LLM : scayle:qwen3
URL : https://172.16.59.5:32443/api

────────────────────────────────────────────────────────────
  Q0: How do historical descriptions of the Dresden's cultural nicknames and the reasons given for them vary across time?
────────────────────────────────────────────────────────────


  1. Route (1.2s): concept = cultural nicknames of Dresden
  ⚠ disambiguation failed: ['A comparison needs at least two slices. Provide two or more periods or source languages.']

────────────────────────────────────────────────────────────
  Q9: How does the definition of a valid marriage shift over time in the Western hemisphere?
────────────────────────────────────────────────────────────


  1. Route (1.2s): concept = marriage definition and societal norms
  ⚠ disambiguation failed: ['A comparison needs at least two slices. Provide two or more periods or source languages.']

────────────────────────────────────────────────────────────
  Q27: What are conceptual differences between love in the early 19th century and today?
────────────────────────────────────────────────────────────


  1. Route (1.0s): concept = love
  2. Disambiguate: concept from LLM → love
  2. Disambiguate: love, temporal, 2 slices
       slice_1: 19th century (1801–1900, English)
       slice_2: present (2026–2026, English)


  3. Extend (6.6s):
       facets: ['romantic love', 'marital love', 'platonic love', 'spiritual love', 'self-love', 'passionate love', 'devotion', 'affection', 'fidelity', 'emotional intimacy']
       lenses: ['conceptual evolution', 'emotional depth', 'social expectations', 'individualism vs. tradition']


  4. Decompose (3.8s):
       #1 How is love defined in the 19th century?
       #2 What terms, expressions, or metaphors are used to describe love in the 19th century?
       #3 How is love defined in the present?
       #4 What terms, expressions, or metaphors are used to describe love in the present?
       #5 How do these accounts differ, given #1, #2, #3, #4?  → [1, 2, 3, 4]

════════════════════════════════════════════════════════════
  Pipeline finished for all benchmark samples.
